# ALMASim download workflow

This notebook demonstrates how to:

1. query ALMA metadata and save it as CSV
2. resolve DataLink products from `member_ous_uid`
3. save the resolved product table as CSV
4. download a filtered subset of products to disk


In [ ]:
from pathlib import Path

from almasim.services.download import (
    download_products,
    filter_products,
    format_bytes,
    resolve_products,
    save_products_csv,
)
from almasim.services.metadata.tap import InclusionFilters, query_metadata_by_science

repo_root = Path.cwd().resolve()
if not (repo_root / "src" / "almasim").exists():
    repo_root = repo_root.parent

output_dir = repo_root / "examples" / "output"
output_dir.mkdir(parents=True, exist_ok=True)
metadata_csv = output_dir / "metadata_query_results.csv"
products_csv = output_dir / "resolved_products.csv"
download_dir = output_dir / "downloads"


In [ ]:
include = InclusionFilters(
    science_keyword=["Galaxies"],
    band=[6],
)
metadata = query_metadata_by_science(include=include).head(5).reset_index(drop=True)
metadata.to_csv(metadata_csv, index=False)

print(f"Saved metadata CSV: {metadata_csv}")
metadata[[col for col in ["ALMA_source_name", "member_ous_uid", "Band"] if col in metadata.columns]].head()


In [ ]:
member_uids = metadata["member_ous_uid"].dropna().astype(str).head(1).tolist()
products = resolve_products(member_uids)
save_products_csv(products, products_csv)

print(f"Saved resolved products CSV: {products_csv}")
print(f"Resolved {len(products)} products")


In [ ]:
filtered = filter_products(products, "all")
print(f"Selected {len(filtered)} products for download: {format_bytes(sum(p.content_length for p in filtered))}")

# Uncomment to perform the actual download in a network-enabled environment.
# summary = download_products(filtered, download_dir, max_parallel=3, extract_tar=False, logger_fn=print)
# summary
